<a href="https://colab.research.google.com/github/Muhammad-Ahmad-1263/code-switching-codesaviours-si26-Muhammad-Ahmad/blob/main/SI26_Week7_MuhammadAhmad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SI26 — Week 7: Train Language ID Model + Deploy

**Project 2 — Code Saviours SI-26**
**Author:** Muhammad Ahmad

This notebook fine-tunes `xlm-roberta-base` for **token-level language identification**
on the Roman Urdu–English code-switched dataset built in Week 6
(`dataset.csv` — 155 sentences, 1,419 labelled words, labels: `URD`, `ENG`, `MIX`).

**Runtime:** Runtime → Change runtime type → **GPU** (T4 is enough for this dataset size).

Pipeline:
1. Load & prepare the dataset
2. Tokenize + align word-level labels to subword tokens
3. Fine-tune XLM-RoBERTa for token classification
4. Evaluate — per-label F1 for URD / ENG / MIX
5. Push the model to the Hugging Face Hub
6. (Optional) Deploy a Streamlit demo on Hugging Face Spaces


In [1]:
!pip install -q transformers torch datasets scikit-learn seqeval huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


## Step 1 — Load and prepare the dataset

A couple of things worth being careful about with this dataset specifically:

- It's **small** (155 sentences) and the label distribution is imbalanced
  (`URD` ≈ 925, `ENG` ≈ 486, `MIX` ≈ 8 tokens) — `MIX` is very rare, so don't be
  surprised if it gets 0 support in some test splits. That's a property of the
  data, not a bug.
- `pandas.groupby('sentence')` sorts groups alphabetically by default. That's
  harmless for training (order doesn't matter once we split), but we pass
  `sort=False` anyway so sentence order stays as in the CSV — makes debugging
  easier if you want to inspect `train_data[0]`.


In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
!wget -q https://raw.githubusercontent.com/Muhammad-Ahmad-1263/code-switching-codesaviours-si26-Muhammad-Ahmad/main/dataset.csv
df = pd.read_csv('dataset.csv')
# Load your dataset
df = pd.read_csv('dataset.csv')
print(f"Rows (word-level): {len(df)}")
print(df['label'].value_counts())

# Create label mapping
label2id = {'URD': 0, 'ENG': 1, 'MIX': 2}
id2label = {0: 'URD', 1: 'ENG', 2: 'MIX'}

# Group by sentence (sort=False keeps original CSV order)
sentences = df.groupby('sentence', sort=False).apply(
    lambda x: {'words': x['word'].tolist(), 'labels': x['label'].tolist()}
).tolist()

print(f"Total sentences: {len(sentences)}")

# Split into train and test
train_data, test_data = train_test_split(sentences, test_size=0.2, random_state=42)
print(f'Training sentences: {len(train_data)}')
print(f'Testing sentences: {len(test_data)}')

Rows (word-level): 1419
label
URD    925
ENG    486
MIX      8
Name: count, dtype: int64
Total sentences: 155
Training sentences: 124
Testing sentences: 31


/tmp/ipykernel_1299/3383474718.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sentences = df.groupby('sentence', sort=False).apply(


In [4]:
# Sanity check: peek at one example
train_data[0]

{'words': ['Kal',
  'ki',
  'flight',
  'delay',
  'ho',
  'gayi',
  'thi',
  'kaafi',
  'ghanton',
  'ke',
  'liye'],
 'labels': ['URD',
  'URD',
  'ENG',
  'ENG',
  'URD',
  'URD',
  'URD',
  'URD',
  'URD',
  'URD',
  'URD']}

## Step 2 — Fine-tune XLM-RoBERTa for token classification

Note on API versions: recent `transformers` releases renamed
`evaluation_strategy` → `eval_strategy` on `TrainingArguments`, and the
`Trainer(tokenizer=...)` argument was replaced with `Trainer(processing_class=...)`.
This notebook uses the current names — if you're on an older `transformers`
version and it errors, swap `eval_strategy` back to `evaluation_strategy` and
`processing_class` back to `tokenizer`.


In [5]:
from transformers import (AutoTokenizer, AutoModelForTokenClassification,
                           TrainingArguments, Trainer, DataCollatorForTokenClassification)
from datasets import Dataset

model_name = 'xlm-roberta-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

def tokenize_and_align_labels(examples):
    tokenized = tokenizer(examples['words'], truncation=True, is_split_into_words=True)
    labels = []
    for i, label in enumerate(examples['labels']):
        word_ids = tokenized.word_ids(batch_index=i)
        label_ids = []
        prev_word = None
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)          # special tokens ([CLS], [SEP], padding)
            elif word_id != prev_word:
                label_ids.append(label2id[label[word_id]])   # first subword of a word gets the label
            else:
                label_ids.append(-100)          # subsequent subwords of the same word are ignored
            prev_word = word_id
        labels.append(label_ids)
    tokenized['labels'] = labels
    return tokenized

def to_hf_dataset(data):
    return Dataset.from_dict({
        'words': [d['words'] for d in data],
        'labels': [d['labels'] for d in data]
    })

train_ds = to_hf_dataset(train_data).map(tokenize_and_align_labels, batched=True)
test_ds = to_hf_dataset(test_data).map(tokenize_and_align_labels, batched=True)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/124 [00:00<?, ? examples/s]

Map:   0%|          | 0/31 [00:00<?, ? examples/s]

### Evaluation metric

The label scheme here (`URD` / `ENG` / `MIX`) is a flat per-word classification,
**not** a BIO/IOB entity-tagging scheme. `seqeval` is built for chunk-level NER
metrics (it expects `B-`/`I-` prefixes) — feeding it flat tags without those
prefixes silently gives you the wrong numbers, since it merges consecutive
identical labels into a single "entity". For this task, plain **per-token**
precision/recall/F1 (via `sklearn.metrics.classification_report`) is the
correct and honest metric, so that's what's used below.


In [6]:
import numpy as np
from sklearn.metrics import classification_report

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    true_labels = []
    true_preds = []
    for pred_row, label_row in zip(predictions, labels):
        for p, l in zip(pred_row, label_row):
            if l == -100:
                continue
            true_labels.append(id2label[l])
            true_preds.append(id2label[p])

    report = classification_report(
        true_labels, true_preds,
        labels=['URD', 'ENG', 'MIX'],
        output_dict=True,
        zero_division=0
    )

    return {
        'f1_URD': report['URD']['f1-score'],
        'f1_ENG': report['ENG']['f1-score'],
        'f1_MIX': report['MIX']['f1-score'],
        'f1_macro': report['macro avg']['f1-score'],
        'f1_weighted': report['weighted avg']['f1-score'],
    }

In [7]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model='f1_weighted',
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
)

print('Starting training...')
trainer.train()
print('Training complete!')

Starting training...


Epoch,Training Loss,Validation Loss,F1 Urd,F1 Eng,F1 Mix,F1 Macro,F1 Weighted
1,No log,0.301490,0.981723,0.941176,0.000000,0.640967,0.958732
2,0.614391,0.114243,0.992042,0.965909,0.000000,0.652651,0.973347
3,0.119933,0.087208,0.994709,0.971429,0.000000,0.655379,0.976857
4,0.040091,0.104810,0.994709,0.971429,0.000000,0.655379,0.976857
5,0.033254,0.101757,0.994709,0.971429,0.000000,0.655379,0.976857


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete!


## Step 3 — Evaluate

This produces the F1 scores for `URD`, `ENG`, and `MIX` you need to paste into
your submission comment.


In [8]:
eval_results = trainer.evaluate()
print(eval_results)

print("\n=== F1 scores for submission ===")
print(f"F1 (URD): {eval_results['eval_f1_URD']:.4f}")
print(f"F1 (ENG): {eval_results['eval_f1_ENG']:.4f}")
print(f"F1 (MIX): {eval_results['eval_f1_MIX']:.4f}")
print(f"Macro F1: {eval_results['eval_f1_macro']:.4f}")
print(f"Weighted F1: {eval_results['eval_f1_weighted']:.4f}")

Training Loss,Validation Loss,Epoch,F1 Urd,F1 Eng,F1 Mix,F1 Macro,F1 Weighted
0.033254,0.087208,5,0.994709,0.971429,0.000000,0.655379,0.976857


{'eval_loss': 0.08720753341913223, 'eval_f1_URD': 0.9947089947089947, 'eval_f1_ENG': 0.9714285714285714, 'eval_f1_MIX': 0.0, 'eval_f1_macro': 0.655379188712522, 'eval_f1_weighted': 0.9768566099501351}

=== F1 scores for submission ===
F1 (URD): 0.9947
F1 (ENG): 0.9714
F1 (MIX): 0.0000
Macro F1: 0.6554
Weighted F1: 0.9769


In [9]:
# Full per-class precision/recall/F1 + confusion, for your own inspection
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

predictions, labels, _ = trainer.predict(test_ds)
predictions = np.argmax(predictions, axis=2)

true_labels, true_preds = [], []
for pred_row, label_row in zip(predictions, labels):
    for p, l in zip(pred_row, label_row):
        if l == -100:
            continue
        true_labels.append(id2label[l])
        true_preds.append(id2label[p])

print(classification_report(true_labels, true_preds, labels=['URD', 'ENG', 'MIX'], zero_division=0))
print("Confusion matrix (rows=true, cols=pred), order URD/ENG/MIX:")
print(confusion_matrix(true_labels, true_preds, labels=['URD', 'ENG', 'MIX']))

              precision    recall  f1-score   support

         URD       1.00      0.99      0.99       190
         ENG       0.94      1.00      0.97        85
         MIX       0.00      0.00      0.00         3

    accuracy                           0.98       278
   macro avg       0.65      0.66      0.66       278
weighted avg       0.97      0.98      0.98       278

Confusion matrix (rows=true, cols=pred), order URD/ENG/MIX:
[[188   2   0]
 [  0  85   0]
 [  0   3   0]]


## Step 4 — Save and push to Hugging Face Hub

Replace `[yourusername]` with your actual Hugging Face username, and make sure
you're logged in with a **write** token (create one at
https://huggingface.co/settings/tokens).


In [11]:
from huggingface_hub import notebook_login
notebook_login()  # paste a HuggingFace token with WRITE access when prompted

In [12]:
repo_name = 'code-switching-codesaviours-si26-muhammadahmad'  # match your Week 6 dataset repo naming

model.push_to_hub(repo_name)
tokenizer.push_to_hub(repo_name)

print(f'Model published — replace [Muhammad Ahmad] with your real HF username:')
print(f'https://huggingface.co/datasets/Muhammad-Ahmad-1263/code-switching-codesaviours-si26-muhammadahmad/{repo_name}')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...e8oqz7z/model.safetensors:   0%|          | 13.6kB / 1.11GB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpc_yxzgdk/tokenizer.json: 100%|##########| 17.1MB / 17.1MB            

Model published — replace [Muhammad Ahmad] with your real HF username:
https://huggingface.co/datasets/Muhammad-Ahmad-1263/code-switching-codesaviours-si26-muhammadahmad/code-switching-codesaviours-si26-muhammadahmad


## Step 5 — Quick sanity check: load the pushed model back and run inference

Fill in your username below once the push above finishes.


In [17]:
from transformers import pipeline

hf_username = "Muhammad-Ahmad-1263"   # <-- fill this in
model_id = f"{hf_username}/{repo_name}"

classifier = pipeline("token-classification", model=model_id, aggregation_strategy=None)

test_sentence = "Yaar mujhe kal ka meeting reschedule karna hai"
results = classifier(test_sentence)
for r in results:
    print(f"{r['word']:15s} -> {r['entity']}  (score={r['score']:.2f})")

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/343 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

▁Yaar           -> URD  (score=1.00)
▁mujhe          -> URD  (score=1.00)
▁kal            -> URD  (score=1.00)
▁ka             -> URD  (score=1.00)
▁meeting        -> ENG  (score=1.00)
▁res            -> ENG  (score=1.00)
ched            -> ENG  (score=1.00)
ule             -> ENG  (score=0.99)
▁karna          -> URD  (score=1.00)
▁hai            -> URD  (score=1.00)


## Step 6 — Deploy a Streamlit demo (Hugging Face Spaces)

1. Go to https://huggingface.co/new-space
2. Pick **Streamlit** as the SDK, name the Space (e.g. `code-switching-lid-demo`)
3. Upload `app.py` and `requirements.txt` (generated alongside this notebook)
   to the Space's file list, or `git push` them to the Space's repo
4. Open the Space's `App` tab once it finishes building — that URL is your demo link

`app.py` loads `hf_username/repo_name` from the Hub and lets a user type a
Roman Urdu / English sentence and see each word tagged URD / ENG / MIX.


## Submission checklist

Paste these in Classroom by Friday:
- [ ] Hugging Face **model** Hub link: `https://huggingface.co/[yourusername]/code-switching-codesaviours-si26-muhammadahmad`
- [ ] This notebook, committed to your Week 7 GitHub repo
- [ ] F1 scores (from the evaluation cell above): URD / ENG / MIX
- [ ] (If deployed) Hugging Face Space / Streamlit demo link
